## QC: Stream Temperature Statistics

Spot-check `wt_stats.nc` by pulling 20 random stream IDs and displaying all statistics in tables. Goal is to verify values are in plausible ranges — not a formal validation.

In [4]:
import xarray as xr
import numpy as np
import pandas as pd
import random
import sys
sys.path.insert(0, '..')
from luts_wt import wt_stat_var_dict

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
# Change this path to the sanity-check output while the full run is pending
WT_STATS_PATH = '/beegfs/CMIP6/jdpaul3/arctic_rivers_data/wt_stats_05282026.nc'

ds = xr.open_dataset(WT_STATS_PATH)
print(ds)

<xarray.Dataset> Size: 506MB
Dimensions:                   (stream_id: 34208, model: 7, era: 2, source: 3)
Coordinates:
  * stream_id                 (stream_id) int64 274kB 81000004 ... 82001714
  * model                     (model) <U10 280B 'C2LE2' 'C2LE4' ... 'historical'
  * era                       (era) <U9 72B '1990-2021' '2034-2065'
  * source                    (source) <U25 300B 'original_gcm' ... 'gcm_diff...
Data variables: (12/44)
    wt_days_gt13_mean         (source, era, model, stream_id) timedelta64[ns] 11MB ...
    wt_days_gt18_mean         (source, era, model, stream_id) timedelta64[ns] 11MB ...
    wt_days_gt20_mean         (source, era, model, stream_id) timedelta64[ns] 11MB ...
    wt_mean_jan               (source, era, model, stream_id) float64 11MB ...
    wt_mean_feb               (source, era, model, stream_id) float64 11MB ...
    wt_mean_mar               (source, era, model, stream_id) float64 11MB ...
    ...                        ...
    wt_max_dec   

In [6]:
random.seed(42)
all_stream_ids = ds.stream_id.values.tolist()
sample_ids = random.sample(all_stream_ids, min(20, len(all_stream_ids)))
print(f'Sampled {len(sample_ids)} stream IDs:', sample_ids)

Sampled 20 stream IDs: [81007640, 81001892, 81018674, 81016686, 81015222, 81009538, 81007044, 81006011, 81028553, 81002342, 81002209, 81006462, 81014892, 81015857, 81035388, 81001992, 81013581, 81028376, 81015018, 81030542]


In [7]:
def stats_table(ds, var_names, stream_ids, model, era, source='original_gcm'):
    """Return a DataFrame: rows = variables, columns = stream_ids."""
    rows = {}
    for var in var_names:
        units = wt_stat_var_dict[var]['units']
        row_label = f'{var}  [{units}]'
        vals = (
            ds[var]
            .sel(model=model, era=era, source=source)
            .sel(stream_id=stream_ids)
            .values
        )
        rows[row_label] = vals
    return pd.DataFrame(rows, index=stream_ids).T

## Threshold exceedance — days above 13 / 18 / 20 °C

**Expected ranges (historical, Alaska streams):** most streams 0–60 days above 13°C; fewer above 18°C; very few or zero above 20°C for cold headwater streams.

In [8]:
thresh_vars = ['wt_days_gt13_mean', 'wt_days_gt18_mean', 'wt_days_gt20_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, thresh_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 1990-2021 ===')
display(stats_table(ds, thresh_vars, sample_ids, 'C2LE2', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect higher counts) ===')
display(stats_table(ds, thresh_vars, sample_ids, 'C2LE2', '2034-2065'))

=== Historical 1990-2021 ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_days_gt13_mean [days],11 days 18:00:00,0 days 00:44:38.400000,3 days 22:30:43.200000,13 days 03:44:38.400000,17 days 00:44:38.400000,6 days 09:00:00,32 days 21:44:38.400000,1 days 00:44:38.400000,0 days,20 days 09:00:00,25 days 09:44:38.400000,1 days 05:15:21.600000,0 days 02:15:21.600000,6 days 23:15:21.600000,0 days,27 days 18:44:38.400000,20 days 10:30:43.200000,0 days,10 days 07:29:16.800000,0 days
wt_days_gt18_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days
wt_days_gt20_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days



=== C2LE2 1990-2021 ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_days_gt13_mean [days],8 days 07:29:16.800000,0 days 14:15:21.600000,7 days 08:15:21.599999999,7 days,8 days 02:15:21.600000,6 days 03:44:38.400000,27 days 10:30:43.200000,1 days 09:00:00,0 days,27 days 14:15:21.600000,38 days 04:30:43.200000,1 days 21:00:00,0 days 06:00:00,4 days 03:00:00,0 days 13:29:16.800000,37 days 12:00:00,10 days 18:00:00,0 days,5 days 07:29:16.800000,0 days
wt_days_gt18_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days
wt_days_gt20_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days



=== C2LE2 2034-2065 (expect higher counts) ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_days_gt13_mean [days],26 days 21:44:38.400000,6 days 03:44:38.400000,25 days 18:00:00,27 days 16:30:43.199999999,31 days 11:15:21.600000,22 days 12:00:00,47 days 00:44:38.400000,8 days 12:00:00,0 days,47 days 21:44:38.400000,57 days 23:15:21.600000,9 days,2 days 03:44:38.400000,20 days 23:15:21.600000,2 days 22:30:43.200000,56 days 11:15:21.600000,35 days 17:15:21.600000,0 days,23 days 03:00:00,0 days 15:00:00
wt_days_gt18_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 12:44:38.400000,1 days 03:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,1 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt20_mean [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00


## Monthly mean temperatures

**Expected pattern:** near 0.1°C in winter (Nov–Mar), warming through spring, peak in July/August, cooling through fall. Should not see values > ~25°C or < 0°C.

In [9]:
mean_vars = [f'wt_mean_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]

print('=== Historical 1990-2021: monthly mean temperatures ===')
display(stats_table(ds, mean_vars, sample_ids, 'historical', '1990-2021'))

=== Historical 1990-2021: monthly mean temperatures ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_mean_jan [degC],2.04,3.23,1.35,2.10,1.90,1.23,1.11,1.47,2.01,2.94,3.16,1.40,1.68,1.94,2.12,2.75,1.90,1.59,2.06,1.99
wt_mean_feb [degC],1.90,2.92,1.32,1.95,1.78,1.19,1.10,1.41,1.81,2.67,2.88,1.35,1.57,1.81,1.91,2.50,1.77,1.43,1.90,1.80
wt_mean_mar [degC],1.91,2.94,1.50,2.02,1.88,1.40,1.26,1.50,1.85,2.75,2.91,1.45,1.65,1.90,1.95,2.62,1.88,1.52,1.96,1.83
wt_mean_apr [degC],2.09,3.08,2.35,2.83,2.64,1.99,2.00,1.82,2.07,3.01,3.07,1.85,2.05,2.66,2.25,3.02,2.74,2.06,2.60,2.15
wt_mean_may [degC],4.27,3.81,4.56,6.39,6.35,4.06,5.24,3.11,2.88,3.98,3.89,3.25,3.47,5.79,3.53,4.25,6.50,3.18,5.87,3.41
wt_mean_jun [degC],10.23,6.49,8.88,10.93,11.10,9.33,11.54,8.18,4.87,6.88,6.62,8.09,7.35,10.26,6.52,8.14,11.44,5.30,10.60,6.31
wt_mean_jul [degC],12.40,9.22,11.39,12.36,12.55,11.82,13.24,11.19,6.91,11.97,12.27,10.97,9.94,11.96,9.17,12.98,12.70,7.35,12.18,8.84
wt_mean_aug [degC],10.89,7.83,10.05,11.42,11.54,10.19,11.62,9.20,6.65,11.09,11.82,9.32,8.60,10.93,9.27,11.61,11.55,7.23,11.11,8.47
wt_mean_sep [degC],6.46,4.97,5.59,8.21,8.17,5.83,7.03,4.87,4.43,7.67,8.20,5.18,4.86,7.50,6.16,7.88,8.10,4.83,7.66,5.25
wt_mean_oct [degC],2.75,3.38,2.31,3.83,3.73,2.35,2.52,2.11,2.55,4.46,4.69,2.22,2.38,3.40,3.31,4.31,3.58,2.57,3.43,2.81


## Monthly min and max temperatures

Min should be at or near 0.1°C (model floor). Max should be the warmest average July/August across all years — slightly above the mean.

In [10]:
min_vars = [f'wt_min_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]
max_vars = [f'wt_max_{m}' for m in ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']]

print('=== Historical 1990-2021: monthly min temperatures ===')
display(stats_table(ds, min_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== Historical 1990-2021: monthly max temperatures ===')
display(stats_table(ds, max_vars, sample_ids, 'historical', '1990-2021'))

=== Historical 1990-2021: monthly min temperatures ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_min_jan [degC],1.95,3.05,0.61,1.92,1.73,0.95,0.80,1.26,1.71,2.73,2.98,1.07,1.45,1.73,1.91,2.38,1.68,1.17,1.90,1.79
wt_min_feb [degC],1.89,2.84,0.96,1.87,1.68,0.90,0.85,1.27,1.67,2.53,2.79,1.15,1.40,1.71,1.80,2.23,1.65,1.04,1.84,1.64
wt_min_mar [degC],1.89,2.87,0.88,1.88,1.75,1.17,0.84,1.28,1.66,2.62,2.84,1.10,1.45,1.74,1.85,2.40,1.67,1.05,1.85,1.69
wt_min_apr [degC],1.94,2.94,1.60,2.07,2.05,1.64,1.58,1.62,1.88,2.85,2.94,1.55,1.71,1.99,1.98,2.76,2.02,1.61,2.03,1.90
wt_min_may [degC],2.61,3.22,3.37,3.71,3.79,2.67,3.08,2.18,2.24,3.23,3.21,2.25,2.35,3.51,2.66,3.27,3.52,2.49,3.42,2.42
wt_min_jun [degC],8.53,5.23,7.63,8.94,9.03,7.30,10.04,6.20,3.88,4.99,4.89,6.35,5.97,8.31,5.28,5.29,9.82,4.45,8.54,5.09
wt_min_jul [degC],10.70,7.36,10.02,10.98,11.14,9.37,11.85,9.34,5.16,7.09,7.00,8.64,8.16,10.63,7.66,8.30,11.32,5.58,10.68,7.29
wt_min_aug [degC],7.63,5.56,8.18,9.59,9.88,6.78,8.70,5.88,4.83,7.49,8.19,6.33,6.13,8.81,7.17,8.09,9.58,5.65,9.11,6.62
wt_min_sep [degC],4.60,4.00,3.81,6.21,6.32,3.77,5.17,3.62,3.04,5.89,6.16,3.58,3.42,5.44,4.46,5.67,5.99,3.36,5.71,3.83
wt_min_oct [degC],2.10,3.02,1.74,2.55,2.54,1.51,1.62,1.63,1.99,3.21,3.36,1.66,1.81,2.26,2.54,2.92,2.30,1.79,2.38,2.11



=== Historical 1990-2021: monthly max temperatures ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_max_jan [degC],2.22,3.61,1.76,2.51,2.21,1.47,1.34,1.64,2.28,3.34,3.58,1.68,1.98,2.29,2.44,3.30,2.27,2.13,2.36,2.42
wt_max_feb [degC],1.92,3.00,1.66,2.10,1.89,1.36,1.36,1.54,2.10,2.81,2.98,1.49,1.75,1.98,2.04,2.68,1.95,1.80,2.02,2.06
wt_max_mar [degC],2.00,3.14,1.89,2.52,2.21,1.72,1.62,1.71,2.04,3.07,3.20,1.74,1.88,2.34,2.14,3.10,2.40,1.96,2.31,2.08
wt_max_apr [degC],2.72,3.26,2.90,3.85,3.48,2.60,2.98,2.12,2.41,3.41,3.40,2.31,2.41,3.57,2.78,3.69,3.81,2.51,3.44,2.72
wt_max_may [degC],6.54,4.95,6.04,8.28,8.20,5.86,7.83,4.62,3.69,5.80,5.29,4.88,5.04,7.78,4.79,6.11,8.85,4.20,7.88,4.72
wt_max_jun [degC],12.02,8.32,11.08,12.41,12.46,11.34,13.03,10.18,6.18,11.22,12.18,10.57,9.30,11.86,8.79,12.90,12.70,6.85,12.13,8.44
wt_max_jul [degC],13.44,11.57,13.22,13.54,13.65,13.41,14.09,12.80,9.88,16.43,17.08,12.60,12.14,13.47,11.78,16.87,13.74,10.40,13.44,11.41
wt_max_aug [degC],12.74,10.43,13.31,13.35,13.36,12.39,13.07,11.38,9.37,14.65,15.45,12.10,11.34,13.19,12.00,15.06,13.38,9.65,13.19,11.42
wt_max_sep [degC],8.15,5.96,7.21,9.79,9.84,7.22,8.90,6.30,5.50,9.91,10.52,6.62,5.89,8.99,8.44,10.66,9.86,5.80,9.32,6.65
wt_max_oct [degC],3.51,4.13,3.30,5.86,5.40,3.36,3.51,2.71,3.75,5.89,5.99,2.92,2.96,5.22,4.31,5.97,5.70,3.69,5.16,3.59


## Annual maximum daily temperature — magnitude and timing

**Magnitude:** should be the highest daily value of the year, above the monthly mean peak. Roughly 10–25°C for most Alaska streams.

**Julian day:** peak should fall in summer — roughly day 150–240 (early June to late August).

In [11]:
ann_vars = ['wt_ann_max_temp_mean', 'wt_ann_max_temp_doy_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, ann_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect slightly higher temps, similar DOY) ===')
display(stats_table(ds, ann_vars, sample_ids, 'C2LE2', '2034-2065'))

=== Historical 1990-2021 ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_ann_max_temp_mean [degC],13.15,10.77,12.91,13.04,13.23,13.01,14.05,12.33,8.30,14.60,15.22,12.39,11.20,12.81,11.01,15.53,13.39,8.81,12.93,10.39
wt_ann_max_temp_doy_mean [day of year],196.91,203.66,205.50,196.47,196.09,197.88,192.62,196.81,211.44,208.94,210.38,199.25,203.25,203.03,210.28,206.72,195.38,208.53,200.44,208.59



=== C2LE2 2034-2065 (expect slightly higher temps, similar DOY) ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_ann_max_temp_mean [degC],13.54,12.85,14.68,13.50,13.67,13.88,14.43,13.08,10.11,16.62,17.10,13.23,12.52,13.39,12.47,17.06,13.80,10.73,13.40,12.10
wt_ann_max_temp_doy_mean [day of year],194.56,198.50,195.06,192.88,193.47,193.66,191.59,195.25,212.91,201.19,201.12,196.69,194.78,194.72,201.78,199.19,191.41,206.94,192.69,206.91


## Maximum 7-day rolling average — magnitude and timing

**Magnitude:** should be slightly lower than the daily max (smoothed). **Julian day:** similar to daily max DOY — the 7-day window centers near the annual peak. Values will differ by a few days from `wt_ann_max_temp_doy_mean`.

In [12]:
roll_vars = ['wt_7d_max_temp_mean', 'wt_7d_max_temp_doy_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, roll_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== Difference: daily max minus 7-day max (expect small positive values) ===')
diff = (
    ds['wt_ann_max_temp_mean'].sel(model='historical', era='1990-2021', source='original_gcm').sel(stream_id=sample_ids)
    - ds['wt_7d_max_temp_mean'].sel(model='historical', era='1990-2021', source='original_gcm').sel(stream_id=sample_ids)
).to_pandas().rename('daily_max - 7d_max  [degC]')
display(diff.to_frame().T)

=== Historical 1990-2021 ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_7d_max_temp_mean [degC],13.03,10.45,12.52,12.93,13.11,12.72,13.83,12.08,8.06,14.16,14.84,12.10,10.97,12.66,10.75,15.06,13.26,8.50,12.81,10.14
wt_7d_max_temp_doy_mean [day of year],197.91,205.25,206.81,198.66,197.50,199.56,194.34,196.31,211.16,209.69,210.38,200.28,202.97,203.97,209.78,206.88,195.94,210.75,200.41,208.56



=== Difference: daily max minus 7-day max (expect small positive values) ===


stream_id,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
daily_max - 7d_max [degC],0.12,0.32,0.38,0.10,0.12,0.29,0.21,0.26,0.24,0.44,0.38,0.29,0.23,0.15,0.26,0.47,0.13,0.30,0.12,0.25


## Cumulative degree days above 0°C (May–September)

**Expected range:** all values > 0 (minimum T_stream is 0.1°C × 153 May–Sept days ≈ 15 °C·days floor). Warm lowland streams could reach ~2000 °C·days. Cold headwater streams near the floor.

In [13]:
cdd_vars = ['wt_cdd_may_sept_mean']

print('=== Historical 1990-2021 ===')
display(stats_table(ds, cdd_vars, sample_ids, 'historical', '1990-2021'))

print('\n=== C2LE2 2034-2065 (expect higher CDD) ===')
display(stats_table(ds, cdd_vars, sample_ids, 'C2LE2', '2034-2065'))

=== Historical 1990-2021 ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_cdd_may_sept_mean [degC days],1354.59,990.06,1239.90,1509.19,1521.73,1262.59,1489.85,1119.82,788.45,1274.61,1311.67,1127.86,1048.49,1421.87,1061.28,1374.62,1539.20,854.25,1451.69,988.86



=== C2LE2 2034-2065 (expect higher CDD) ===


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_cdd_may_sept_mean [degC days],1517.16,1180.61,1486.58,1652.85,1659.08,1465.24,1641.18,1288.80,990.04,1593.88,1653.96,1312.60,1259.85,1578.85,1334.07,1668.37,1669.46,1084.64,1592.17,1277.27


## NaN structure checks

Verify the expected NaN patterns from the `source` dimension:
- `historical` model has no future projections → all-NaN in the `2034-2065` era for `original_gcm`
- `PGWh`/`PGWm` are future-only → all-NaN in the `1990-2021` era for `original_gcm`
- `gcm_diff` past era should always be all-NaN (by design)
- GCM models (C2LE*) should have data in both eras

In [14]:
sid = sample_ids[0]
check_var = 'wt_ann_max_temp_mean'

checks = {
    'historical / 2034-2065 / original_gcm  (expect NaN)':  ds[check_var].sel(model='historical', era='2034-2065', source='original_gcm', stream_id=sid).item(),
    'PGWh      / 1990-2021 / original_gcm  (expect NaN)':   ds[check_var].sel(model='PGWh',       era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'PGWm      / 1990-2021 / original_gcm  (expect NaN)':   ds[check_var].sel(model='PGWm',       era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'any model / 1990-2021 / gcm_diff      (expect NaN)':   ds[check_var].sel(model='C2LE2',      era='1990-2021', source='gcm_diff',      stream_id=sid).item(),
    'historical / 1990-2021 / original_gcm (expect value)': ds[check_var].sel(model='historical', era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 1990-2021 / original_gcm  (expect value)': ds[check_var].sel(model='C2LE2',      era='1990-2021', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 2034-2065 / original_gcm  (expect value)': ds[check_var].sel(model='C2LE2',      era='2034-2065', source='original_gcm', stream_id=sid).item(),
    'C2LE2     / 2034-2065 / gcm_diff      (expect value)': ds[check_var].sel(model='C2LE2',      era='2034-2065', source='gcm_diff',      stream_id=sid).item(),
}

for label, val in checks.items():
    is_nan = np.isnan(val)
    expected_nan = 'expect NaN' in label
    status = 'OK' if is_nan == expected_nan else 'FAIL'
    print(f'[{status}]  {label}  →  {val}')

[OK]  historical / 2034-2065 / original_gcm  (expect NaN)  →  nan
[OK]  PGWh      / 1990-2021 / original_gcm  (expect NaN)  →  nan
[OK]  PGWm      / 1990-2021 / original_gcm  (expect NaN)  →  nan
[OK]  any model / 1990-2021 / gcm_diff      (expect NaN)  →  nan
[OK]  historical / 1990-2021 / original_gcm (expect value)  →  13.15
[OK]  C2LE2     / 1990-2021 / original_gcm  (expect value)  →  12.954
[OK]  C2LE2     / 2034-2065 / original_gcm  (expect value)  →  13.542
[OK]  C2LE2     / 2034-2065 / gcm_diff      (expect value)  →  0.5879999999999992



## original_gcm vs gcm_diff_applied_to_blaskey (2034–2065)

`gcm_diff_applied_to_blaskey` applies each GCM's delta (future − past) to the historical Blaskey baseline rather than using the raw GCM future value:
```
gcm_diff_applied_to_blaskey = historical_past + (gcm_future − gcm_past)
```
This comparison checks how much the two approaches diverge. Large `orig-adj` values mean the GCM and the historical Blaskey run disagree on the baseline, so the choice of source matters for the front end. Only GCM models (C2LE*) have non-NaN values for both sources in the future era.


In [ ]:

GCM_MODELS = ['C2LE2', 'C2LE4', 'C2LE7', 'C2LE9']

def comparison_table(ds, var_names, stream_ids, model, era='2034-2065'):
    """Side-by-side comparison of original_gcm vs gcm_diff_applied_to_blaskey.
    Returns a DataFrame with three row blocks per variable:
      - original   (raw GCM future)
      - adjusted   (historical Blaskey baseline + GCM delta)
      - orig-adj   (difference; positive = GCM runs warmer than bias-corrected)
    """
    rows = {}
    for var in var_names:
        units = wt_stat_var_dict[var]['units']
        orig = ds[var].sel(model=model, era=era, source='original_gcm').sel(stream_id=stream_ids).values
        adj  = ds[var].sel(model=model, era=era, source='gcm_diff_applied_to_blaskey').sel(stream_id=stream_ids).values
        rows[f'{var}  original  [{units}]'] = orig
        rows[f'{var}  adjusted  [{units}]'] = adj
        rows[f'{var}  orig-adj  [{units}]'] = orig - adj
    return pd.DataFrame(rows, index=stream_ids).T


In [16]:

stat_groups = [
    ('Threshold exceedance (days)',        thresh_vars),
    ('Monthly mean temperatures',          mean_vars),
    ('Annual max + 7-day rolling max',     ann_vars + roll_vars),
    ('Cumulative degree days (May–Sept)',   cdd_vars),
]

for model in GCM_MODELS:
    print(f'\n{"="*70}')
    print(f'  MODEL: {model}  |  era: 2034-2065')
    print(f'{"="*70}')
    for group_label, var_list in stat_groups:
        print(f'\n--- {group_label} ---')
        print('  rows: [variable  original], [variable  adjusted], [variable  orig-adj]')
        display(comparison_table(ds, var_list, sample_ids, model))



  MODEL: C2LE2  |  era: 2034-2065

--- Threshold exceedance (days) ---
  rows: [variable  original], [variable  adjusted], [variable  orig-adj]


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_days_gt13_mean original [days],26 days 21:44:38.400000,6 days 03:44:38.400000,25 days 18:00:00,27 days 16:30:43.199999999,31 days 11:15:21.600000,22 days 12:00:00,47 days 00:44:38.400000,8 days 12:00:00,0 days,47 days 21:44:38.400000,57 days 23:15:21.600000,9 days 00:00:00,2 days 03:44:38.400000,20 days 23:15:21.600000,2 days 22:30:43.200000,56 days 11:15:21.600000,35 days 17:15:21.600000,0 days,23 days 03:00:00,0 days 15:00:00
wt_days_gt13_mean adjusted [days],30 days 08:15:21.599999999,5 days 14:13:55.200000,22 days 08:15:21.599999999,33 days 20:15:21.600000,40 days 09:44:38.400000,22 days 17:15:21.600000,52 days 11:58:33.600000,8 days 03:44:38.400000,0 days,40 days 16:29:16.800000,45 days 04:29:16.800000,8 days 08:15:21.599999999,2 days 00:00:00,23 days 19:30:43.199999999,2 days 09:01:26.400000,46 days 18:00:00,45 days 09:46:04.799999999,0 days,28 days 03:00:00,0 days 15:00:00
wt_days_gt13_mean orig-adj [days],-4 days +13:29:16.800000001,0 days 13:30:43.200000,3 days 09:44:38.400000001,-7 days +20:15:21.599999999,-9 days +01:30:43.200000,-1 days +18:44:38.400000,-6 days +12:46:04.800000,0 days 08:15:21.600000,0 days,7 days 05:15:21.600000,12 days 18:46:04.800000,0 days 15:44:38.400000001,0 days 03:44:38.400000,-3 days +03:44:38.400000001,0 days 13:29:16.800000,9 days 17:15:21.600000,-10 days +07:29:16.800000001,0 days,-5 days +00:00:00,0 days 00:00:00
wt_days_gt18_mean original [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 12:44:38.400000,1 days 03:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,1 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt18_mean adjusted [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 12:44:38.400000,1 days 03:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,1 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt18_mean orig-adj [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt20_mean original [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt20_mean adjusted [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00
wt_days_gt20_mean orig-adj [days],0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days,0 days 00:00:00,0 days 00:00:00



--- Monthly mean temperatures ---
  rows: [variable  original], [variable  adjusted], [variable  orig-adj]


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_mean_jan original [degC],2.12,3.40,1.61,2.35,2.12,1.40,1.34,1.61,2.21,3.21,3.38,1.61,1.80,2.18,2.18,3.10,2.16,1.85,2.25,2.14
wt_mean_jan adjusted [degC],2.09,3.31,1.41,2.26,2.05,1.33,1.20,1.52,2.15,3.17,3.35,1.45,1.74,2.09,2.16,3.00,2.04,1.74,2.18,2.09
wt_mean_jan orig-adj [degC],0.03,0.09,0.20,0.08,0.07,0.07,0.15,0.09,0.07,0.03,0.03,0.16,0.06,0.09,0.02,0.10,0.12,0.11,0.07,0.05
wt_mean_feb original [degC],1.94,3.02,1.52,2.16,1.94,1.33,1.28,1.51,1.99,2.81,2.99,1.50,1.66,2.01,1.96,2.71,1.99,1.68,2.05,1.93
wt_mean_feb adjusted [degC],1.92,2.98,1.38,2.07,1.86,1.30,1.18,1.47,1.90,2.74,2.93,1.44,1.61,1.92,1.94,2.60,1.90,1.54,1.99,1.85
wt_mean_feb orig-adj [degC],0.02,0.05,0.14,0.09,0.08,0.03,0.09,0.04,0.09,0.07,0.06,0.06,0.05,0.09,0.03,0.10,0.09,0.13,0.07,0.08
wt_mean_mar original [degC],1.97,3.03,1.78,2.35,2.13,1.53,1.55,1.64,2.05,2.88,3.02,1.62,1.78,2.21,2.04,2.82,2.19,1.81,2.20,2.01
wt_mean_mar adjusted [degC],1.93,2.98,1.55,2.12,1.96,1.47,1.35,1.56,1.91,2.84,2.99,1.51,1.68,2.00,1.96,2.75,1.98,1.57,2.03,1.85
wt_mean_mar orig-adj [degC],0.04,0.05,0.24,0.23,0.17,0.06,0.20,0.08,0.15,0.04,0.04,0.11,0.10,0.21,0.08,0.07,0.22,0.23,0.17,0.16
wt_mean_apr original [degC],2.43,3.28,2.85,3.66,3.33,2.43,2.58,2.08,2.36,3.35,3.40,2.16,2.35,3.36,2.58,3.46,3.52,2.46,3.28,2.57



--- Annual max + 7-day rolling max ---
  rows: [variable  original], [variable  adjusted], [variable  orig-adj]


...

,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_mean_jan original [degC],2.15,3.48,1.66,2.43,2.19,1.46,1.44,1.68,2.25,3.31,3.47,1.69,1.83,2.26,2.19,3.24,2.25,1.90,2.31,2.17
wt_mean_jan adjusted [degC],2.12,3.38,1.51,2.34,2.11,1.42,1.34,1.62,2.18,3.27,3.45,1.58,1.79,2.16,2.18,3.15,2.14,1.81,2.24,2.11
wt_mean_jan orig-adj [degC],0.03,0.10,0.14,0.10,0.07,0.04,0.10,0.06,0.07,0.03,0.01,0.11,0.04,0.09,0.02,0.09,0.10,0.09,0.07,0.05
wt_mean_feb original [degC],1.95,3.05,1.54,2.15,1.94,1.34,1.27,1.52,2.01,2.81,2.98,1.49,1.68,2.00,1.99,2.73,2.00,1.72,2.05,1.95
wt_mean_feb adjusted [degC],1.93,2.99,1.41,2.09,1.88,1.27,1.16,1.45,1.93,2.75,2.94,1.42,1.63,1.93,1.98,2.64,1.91,1.61,2.00,1.90
wt_mean_feb orig-adj [degC],0.02,0.06,0.14,0.07,0.06,0.07,0.10,0.06,0.08,0.06,0.04,0.07,0.05,0.07,0.01,0.10,0.09,0.11,0.05,0.05
wt_mean_mar original [degC],2.01,3.09,1.91,2.53,2.27,1.63,1.61,1.67,2.10,2.92,3.06,1.68,1.85,2.35,2.13,2.90,2.35,1.92,2.32,2.12
wt_mean_mar adjusted [degC],1.99,3.05,1.77,2.38,2.15,1.60,1.46,1.60,1.99,2.87,3.02,1.60,1.78,2.21,2.09,2.83,2.20,1.75,2.21,2.02
wt_mean_mar orig-adj [degC],0.02,0.04,0.14,0.15,0.11,0.03,0.15,0.06,0.11,0.05,0.04,0.08,0.07,0.14,0.04,0.07,0.15,0.17,0.11,0.10
wt_mean_apr original [degC],2.40,3.27,2.81,3.83,3.48,2.38,2.52,2.03,2.37,3.26,3.31,2.13,2.37,3.48,2.61,3.36,3.60,2.44,3.38,2.58



--- Annual max + 7-day rolling max ---
  rows: [variable  original], [variable  adjusted], [variable  orig-adj]


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_ann_max_temp_mean original [degC],13.50,12.75,14.56,13.45,13.63,13.77,14.42,13.03,9.90,16.74,17.09,13.08,12.39,13.32,12.60,17.07,13.75,10.81,13.36,12.23
wt_ann_max_temp_mean adjusted [degC],13.64,12.92,14.42,13.61,13.83,13.82,14.57,13.30,10.73,16.21,16.59,13.21,12.50,13.58,12.54,16.90,13.97,11.28,13.60,11.92
wt_ann_max_temp_mean orig-adj [degC],-0.14,-0.17,0.15,-0.16,-0.20,-0.06,-0.15,-0.27,-0.82,0.52,0.50,-0.13,-0.10,-0.26,0.06,0.17,-0.22,-0.47,-0.24,0.31
wt_ann_max_temp_doy_mean original [day of year],195.31,199.81,198.41,194.75,194.16,195.22,192.19,195.78,210.94,195.28,197.31,196.53,197.00,197.09,213.00,193.47,192.69,209.03,194.34,210.22
wt_ann_max_temp_doy_mean adjusted [day of year],192.16,201.34,202.22,198.47,198.41,190.84,191.00,193.72,218.56,197.47,200.53,195.34,197.97,208.00,218.59,198.47,197.41,212.84,203.19,217.03
wt_ann_max_temp_doy_mean orig-adj [day of year],3.16,-1.53,-3.81,-3.72,-4.25,4.38,1.19,2.06,-7.63,-2.19,-3.22,1.19,-0.97,-10.91,-5.59,-5.00,-4.72,-3.81,-8.84,-6.81
wt_7d_max_temp_mean original [degC],13.41,12.39,14.09,13.39,13.55,13.49,14.22,12.84,9.70,16.38,16.80,12.82,12.22,13.22,12.44,16.67,13.66,10.49,13.29,12.04
wt_7d_max_temp_mean adjusted [degC],13.54,12.49,13.98,13.55,13.74,13.52,14.37,13.06,10.47,15.79,16.23,12.91,12.35,13.47,12.36,16.40,13.88,10.97,13.52,11.74
wt_7d_max_temp_mean orig-adj [degC],-0.14,-0.10,0.11,-0.16,-0.20,-0.03,-0.14,-0.23,-0.77,0.59,0.57,-0.09,-0.13,-0.25,0.08,0.28,-0.22,-0.48,-0.23,0.30
wt_7d_max_temp_doy_mean original [day of year],195.12,199.41,196.12,194.38,193.94,194.75,191.50,195.06,211.94,197.06,196.53,196.50,197.38,193.16,213.31,194.66,193.16,208.66,193.56,208.59



--- Cumulative degree days (May–Sept) ---
  rows: [variable  original], [variable  adjusted], [variable  orig-adj]


,81007640,81001892,81018674,81016686,81015222,81009538,81007044,81006011,81028553,81002342,81002209,81006462,81014892,81015857,81035388,81001992,81013581,81028376,81015018,81030542
wt_cdd_may_sept_mean original [degC days],1512.10,1180.31,1481.66,1665.82,1674.19,1454.53,1637.27,1290.18,976.51,1628.11,1685.61,1301.15,1257.40,1589.08,1339.33,1697.09,1678.38,1075.19,1605.69,1280.26
wt_cdd_may_sept_mean adjusted [degC days],1545.68,1208.02,1477.53,1721.62,1733.75,1468.84,1677.06,1332.31,1032.78,1556.21,1591.71,1324.52,1285.30,1649.05,1318.40,1645.56,1754.70,1108.69,1674.62,1238.40
wt_cdd_may_sept_mean orig-adj [degC days],-33.58,-27.71,4.12,-55.80,-59.56,-14.31,-39.79,-42.12,-56.27,71.90,93.90,-23.38,-27.90,-59.97,20.93,51.52,-76.32,-33.50,-68.93,41.86


In [ ]:

# Cross-model summary: orig − adj for annual max temperature
# One row per GCM model, columns are the 20 sampled stream IDs.
# Shows at a glance whether divergence between sources is consistent across models.
summary_rows = {}
for model in GCM_MODELS:
    orig = ds['wt_ann_max_temp_mean'].sel(model=model, era='2034-2065', source='original_gcm').sel(stream_id=sample_ids).values
    adj  = ds['wt_ann_max_temp_mean'].sel(model=model, era='2034-2065', source='gcm_diff_applied_to_blaskey').sel(stream_id=sample_ids).values
    summary_rows[model] = orig - adj

summary_df = pd.DataFrame(summary_rows, index=sample_ids).T
summary_df.index.name = 'model'
print('orig − adj for wt_ann_max_temp_mean  [degC]  (2034-2065)')
display(summary_df)
